# 12 Small-class boxless INSTANCE segmentation (center+offset) + YOLO-large hybrid (baseline)

**Route B baseline.** src/11 showed boxless *semantic* segmentation does not beat V6's loose-box detector
on the small classes. Its two structural holes were: (1) **connected components** conflate same-class
pixel connectivity with object identity (touching caries merge, fragments split), and (2) **mean class
probability** is not a ranking score, so the mAP PR-curve is mis-ordered. This notebook replaces *only
those two pieces* with **learned instance grouping + a learned score** (Panoptic-DeepLab-style
center heatmap + offset-to-center). Everything else is held identical to src/11.

**Single-variable discipline (deliberate):**
- **Same as src/11:** multiclass semantic head, fixed `512x1024` resize, `BG_WEIGHT=0.2`, the
  comparable Mask mAP (src/03/04/09), and the **LARGE -> V6** routing (Abrasion/Crown/Filling untouched).
- **The only change:** small-class instances come from a **center+offset** branch (grouping) with the
  **center-heatmap peak as the confidence** (score), instead of `connectedComponents` + mean-prob.
- Deferred to later single-variable runs (NOT in this baseline): binary caries-vs-bg + subtype head,
  higher resolution / tiling, Dice/Focal semantic loss, StarDist grouping.

**How the new branch works.** One shared `U-Net(resnet18)` outputs `N_SEM + 3` channels: `N_SEM`
semantic logits, `1` center-heatmap logit, `2` offset (dx,dy). At decode: peak-detect the center
heatmap -> instance centers (+score); every foreground pixel votes `(x+dx, y+dy)` and is assigned to the
nearest center -> instance pixel groups (this splits touching same-class lesions); instance class =
majority semantic label of its pixels; instance score = center peak value.

**Pre-registered reading (same headline as src/11, so the two are directly comparable):**
- **Headline = supported-small instance-seg AP vs V6 small AP** (Caries 1/2/3/5 only; 4/6 are noise).
  Does learned grouping + learned score beat src/11's connected-components + mean-prob, and beat V6?
- Secondary = **hybrid aggregate (9 classes) vs V6 0.2099** (conservative; small classes are low-weight).
- **Go**: headline clearly > src/11 and > V6 (beyond ~0.003) -> route B works; refine (binary+subtype,
  higher res, better checkpoint metric, TTA). **No-go**: flat/negative -> the small-class deficit is the
  pixel signal (resolution), not the instance/score machinery -> next lever is resolution/tiling, or stop.

**Inputs (Kaggle):** training dataset with `yolo_seg_train.yaml` (train+val images **and** labels) +
the **V6** detector (`version6...best.pt`). `segmentation_models_pytorch` + `ultralytics` (net allowed).

## 1. Setup

In [ ]:
import importlib.util, subprocess, sys
for pkg, pipname in [("segmentation_models_pytorch", "segmentation-models-pytorch"),
                     ("ultralytics", "ultralytics")]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pipname])
print("deps ok")

In [ ]:
import os, json, random
from pathlib import Path
import numpy as np
import cv2
import yaml
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from tqdm.auto import tqdm
import segmentation_models_pytorch as smp
from ultralytics import YOLO

cv2.setNumThreads(0)
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

## 2. Configuration

`LARGE_NAMES` route to YOLO; the caries classes route to the instance segmenter. The model sees a fixed
`IMG_H x IMG_W` resize (panoramic ~2:1). Normalized polygon coords are resize-invariant, so predicted
instances map straight back to full-image space for scoring.

The instance knobs (heatmap / offset / peak) are the only new config vs src/11.

In [ ]:
# --- Hybrid routing ---
LARGE_NAMES = {"abrasion", "crown", "filling"}   # -> YOLO; all other (caries) classes -> instance seg

# --- Model / training (IDENTICAL to src/11 for single-variable comparison) ---
ENCODER      = "resnet18"
ENCODER_WTS  = "imagenet"
IMG_H, IMG_W = 512, 1024      # fixed input (multiples of 32); ~matches panoramic aspect
EPOCHS       = 40
BATCH_SIZE   = 4
LR           = 1e-3
BG_WEIGHT    = 0.2            # CE weight on background (vs 1.0 per caries class) — severe class imbalance

# --- Instance branch (center + offset) — the new machinery vs src/11 ---
HEATMAP_SIGMA   = 4.0         # gaussian radius (px @ 512x1024) splatted at each instance centroid
OFFSET_NORM     = 128.0       # offsets are stored/predicted in units of this many px (keeps L1 ~O(1))
PEAK_THRESH     = 0.30        # center-heatmap peak detection threshold
PEAK_NMS_KERNEL = 5           # max-pool kernel for local-maximum (NMS) peak detection (odd)
LAMBDA_CENTER   = 1.0         # weight on the center focal loss
LAMBDA_OFFSET   = 0.1         # weight on the offset L1 loss
MIN_COMPONENT_PX = 12         # drop instances with fewer assigned pixels than this (noise)
CAPTURE_CONF     = 0.001      # low: mask-AP needs the full PR curve (applies to the YOLO branch)

# --- Eval (identical to src/03/04/09) ---
IOU_THRESHOLDS = np.linspace(0.5, 0.95, 10)
V6_REF_AP = {"abrasion":0.6471,"crown":0.6313,"filling":0.2799,"caries 1":0.1195,
             "caries 2":0.0845,"caries 3":0.0116,"caries 4":0.0000,"caries 5":0.1097,"caries 6":0.0051}
V6_REF_MAP = 0.2099
SEMSEG_REF_SMALL = {"caries 1":0.0550,"caries 2":0.0204,"caries 3":0.0046,"caries 5":0.0473}  # src/11 (version16)
SUPPORTED_SMALL = {"caries 1","caries 2","caries 3","caries 5"}   # 4 (n=4) / 6 (n=5) are noise

def _norm_class_key(nm):
    s = str(nm).lower().replace("class", "")
    return " ".join(s.split())

## 3. Dataset + ground truth (train + val splits)

In [ ]:
INPUT_ROOT = Path("/kaggle/input")
yc = list(INPUT_ROOT.rglob("yolo_seg_train.yaml"))
if not yc:
    raise FileNotFoundError("No yolo_seg_train.yaml under /kaggle/input (attach the training dataset).")
DATA_YAML = yc[0]
dcfg = yaml.safe_load(open(DATA_YAML, encoding="utf-8"))
names = dcfg.get("names")
if isinstance(names, dict): names = [names[k] for k in sorted(names)]
num_classes = len(names)
dataset_root = DATA_YAML.parent
yaml_root = Path(dcfg["path"]) if dcfg.get("path") else dataset_root
if not yaml_root.is_absolute(): yaml_root = dataset_root / yaml_root

def resolve_split(v):
    if v is None: return None
    p = Path(v)
    if p.is_absolute(): return p
    for c in (yaml_root / p, dataset_root / p):
        if c.exists(): return c
    return yaml_root / p

train_images = resolve_split(dcfg.get("train"))
val_images   = resolve_split(dcfg.get("val", dcfg.get("valid")))

def labels_dir_for(images_dir):
    parts = list(Path(images_dir).parts)
    if "images" in parts:
        i = len(parts) - 1 - parts[::-1].index("images"); parts[i] = "labels"
        return Path(*parts)
    return Path(images_dir).parent / "labels"

IMG_EXTS = {".jpg",".jpeg",".png",".bmp",".webp",".tif",".tiff"}
def parse_seg_line(line):
    parts = line.strip().split()
    if len(parts) < 7: return None
    try: cls = int(float(parts[0])); coords = [float(v) for v in parts[1:]]
    except ValueError: return None
    if len(coords) % 2: coords = coords[:-1]
    poly = np.asarray(coords, dtype=np.float64).reshape(-1, 2)
    return (cls, poly) if len(poly) >= 3 else None

def load_objs(images_dir):
    objs = {}; ldir = labels_dir_for(images_dir)
    for ip in sorted(p for p in Path(images_dir).iterdir() if p.suffix.lower() in IMG_EXTS):
        lp = ldir / (ip.stem + ".txt"); o = []
        if lp.exists():
            for line in lp.read_text().splitlines():
                pr = parse_seg_line(line)
                if pr is not None: o.append(pr)
        objs[str(ip)] = o
    return objs

train_objs = load_objs(train_images)
val_objs   = load_objs(val_images)

LARGE_IDX = [c for c in range(num_classes) if _norm_class_key(names[c]) in LARGE_NAMES]
SMALL_IDX = [c for c in range(num_classes) if c not in LARGE_IDX]   # caries -> instance seg
small_to_sem = {c: i + 1 for i, c in enumerate(SMALL_IDX)}          # 0 = background
sem_to_small = {v: k for k, v in small_to_sem.items()}
N_SEM = len(SMALL_IDX) + 1
print("classes:", names)
print("LARGE (YOLO):", [names[c] for c in LARGE_IDX])
print("SMALL (instance seg):", [names[c] for c in SMALL_IDX], "-> sem channels:", N_SEM)
print("train images:", len(train_objs), "| val images:", len(val_objs))

## 4. Instance targets — semantic map + center heatmap + offset field

For each small-class GT polygon (one instance) we rasterize its mask in the resized frame, compute its
centroid, splat a gaussian into the **center heatmap**, and fill the **offset field** (each instance
pixel -> vector to its own centroid, in `OFFSET_NORM`-px units). `offset_mask` marks the foreground
pixels where the offset loss applies. The semantic map is the src/11 class map. hflip is applied to the
**polygons first** (`x -> 1-x`) so every target is generated consistently (no offset-sign bookkeeping).

In [ ]:
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def load_image_resized(ip):
    img = cv2.imread(ip, cv2.IMREAD_COLOR)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return cv2.resize(img, (IMG_W, IMG_H), interpolation=cv2.INTER_LINEAR)

def _draw_gaussian(hm, cx, cy, sigma):
    H, W = hm.shape; r = int(3 * sigma)
    x0, x1 = max(0, cx - r), min(W, cx + r + 1)
    y0, y1 = max(0, cy - r), min(H, cy + r + 1)
    if x1 <= x0 or y1 <= y0: return
    ys, xs = np.ogrid[y0:y1, x0:x1]
    g = np.exp(-((xs - cx) ** 2 + (ys - cy) ** 2) / (2.0 * sigma * sigma)).astype(np.float32)
    hm[y0:y1, x0:x1] = np.maximum(hm[y0:y1, x0:x1], g)

def make_targets(objs, H, W):
    # NOTE: int32 (CV_32S) for cv2.fillPoly — int64 is rejected ("output array incompatible with cv::Mat").
    sem   = np.zeros((H, W), dtype=np.int32)     # 0=bg, else small_to_sem (later instances overwrite — ok baseline)
    hm    = np.zeros((H, W), dtype=np.float32)   # center heatmap (gaussian peaks)
    off   = np.zeros((2, H, W), dtype=np.float32)  # offset-to-centroid (dx,dy) in OFFSET_NORM-px units
    omask = np.zeros((H, W), dtype=np.float32)   # where the offset loss applies (instance fg)
    for cls, poly in objs:
        if cls not in small_to_sem: continue
        pts = poly.copy(); pts[:, 0] *= W; pts[:, 1] *= H
        inst = np.zeros((H, W), dtype=np.uint8)
        cv2.fillPoly(inst, [pts.astype(np.int32)], 1)
        ys, xs = np.where(inst > 0)
        if len(xs) == 0: continue
        cx = float(xs.mean()); cy = float(ys.mean())
        sem[ys, xs] = int(small_to_sem[cls])
        _draw_gaussian(hm, int(round(cx)), int(round(cy)), HEATMAP_SIGMA)
        off[0, ys, xs] = (cx - xs) / OFFSET_NORM
        off[1, ys, xs] = (cy - ys) / OFFSET_NORM
        omask[ys, xs] = 1.0
    return sem, hm, off, omask

class InstDataset(torch.utils.data.Dataset):
    def __init__(self, objs_dict, augment=False):
        self.items = list(objs_dict.items()); self.augment = augment
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        ip, objs = self.items[i]
        img = load_image_resized(ip)
        polys = objs
        if self.augment and random.random() < 0.5:                 # hflip on image AND polygons (x->1-x)
            img = img[:, ::-1, :].copy()
            polys = [(c, np.column_stack([1.0 - p[:, 0], p[:, 1]])) for c, p in objs]
        sem, hm, off, omask = make_targets(polys, IMG_H, IMG_W)
        x = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
        x = (x - IMAGENET_MEAN) / IMAGENET_STD
        return (x, torch.from_numpy(sem).long(), torch.from_numpy(hm),
                torch.from_numpy(off), torch.from_numpy(omask))

train_loader = torch.utils.data.DataLoader(InstDataset(train_objs, augment=True),
                                           batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)
val_loader   = torch.utils.data.DataLoader(InstDataset(val_objs, augment=False),
                                           batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print("train/val datasets:", len(train_loader.dataset), len(val_loader.dataset))

## 5. Train (shared U-Net -> semantic + center + offset; checkpoint by val fg-mIoU)

One `U-Net` outputs `N_SEM + 3` channels, split into semantic logits / center logit / offset.
Losses: weighted CE (semantic) + CenterNet penalty-reduced focal (center heatmap) + masked L1 (offset).

> **Decision flagged:** checkpoint is still **val fg-mIoU** (semantic proxy, same as src/11) for a clean
> comparison. It does **not** measure instance quality; if the result is borderline, a center-detection
> AP proxy is the natural upgrade. Raised with the user.

In [ ]:
model = smp.Unet(encoder_name=ENCODER, encoder_weights=ENCODER_WTS, in_channels=3, classes=N_SEM + 3).to(DEVICE)
ce_weight = torch.tensor([BG_WEIGHT] + [1.0] * (N_SEM - 1), dtype=torch.float32, device=DEVICE)
sem_criterion = nn.CrossEntropyLoss(weight=ce_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

def split_outputs(out):
    # out: (B, N_SEM+3, H, W) -> semantic logits (B,N_SEM,H,W), center logit (B,H,W), offset (B,2,H,W)
    return out[:, :N_SEM], out[:, N_SEM], out[:, N_SEM + 1:N_SEM + 3]

def center_focal_loss(pred, gt):
    # CenterNet penalty-reduced focal. pred = sigmoid in (0,1), gt in [0,1] (peak == 1 at centroids).
    pred = pred.clamp(1e-6, 1.0 - 1e-6)
    pos = (gt >= 1.0).float(); neg = 1.0 - pos
    neg_w = torch.pow(1.0 - gt, 4)
    pos_loss = (torch.log(pred) * torch.pow(1.0 - pred, 2) * pos).sum()
    neg_loss = (torch.log(1.0 - pred) * torch.pow(pred, 2) * neg_w * neg).sum()
    n = pos.sum()
    return -(pos_loss + neg_loss) / n if n > 0 else -neg_loss

def offset_l1_loss(pred, gt, mask):
    m = mask.unsqueeze(1)                                  # (B,1,H,W)
    return (torch.abs(pred - gt) * m).sum() / (m.sum() * 2.0 + 1e-6)

@torch.no_grad()
def val_fg_miou():
    model.eval(); inter = np.zeros(N_SEM); union = np.zeros(N_SEM)
    for x, y, *_ in val_loader:
        sem_l, _, _ = split_outputs(model(x.to(DEVICE)))
        pred = sem_l.argmax(1).cpu().numpy(); y = y.numpy()
        for cl in range(1, N_SEM):
            p = (pred == cl); g = (y == cl)
            inter[cl] += int((p & g).sum()); union[cl] += int((p | g).sum())
    ious = [inter[cl] / union[cl] for cl in range(1, N_SEM) if union[cl] > 0]
    return float(np.mean(ious)) if ious else 0.0

best_iou, best_state = -1.0, None
for ep in range(1, EPOCHS + 1):
    model.train(); tl = ts = tc = to = 0.0
    for x, sem, hm, off, om in train_loader:
        optimizer.zero_grad()
        sem_l, ctr_l, off_p = split_outputs(model(x.to(DEVICE)))
        L_sem = sem_criterion(sem_l, sem.to(DEVICE))
        L_ctr = center_focal_loss(torch.sigmoid(ctr_l), hm.to(DEVICE))
        L_off = offset_l1_loss(off_p, off.to(DEVICE), om.to(DEVICE))
        loss = L_sem + LAMBDA_CENTER * L_ctr + LAMBDA_OFFSET * L_off
        loss.backward(); optimizer.step()
        tl += loss.item(); ts += L_sem.item(); tc += L_ctr.item(); to += L_off.item()
    viou = val_fg_miou(); nb = len(train_loader)
    if viou > best_iou:
        best_iou = viou; best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    print(f"ep {ep:3d}  loss {tl/nb:.4f} (sem {ts/nb:.4f} ctr {tc/nb:.4f} off {to/nb:.4f})  "
          f"val_fg_mIoU {viou:.4f}  best {best_iou:.4f}")
if best_state is not None: model.load_state_dict(best_state)
print(f"loaded best (val_fg_mIoU={best_iou:.4f})")

## 6. Decode -> small-class instances (center peaks + offset voting)

Peak-detect the center heatmap (max-pool NMS) -> instance centers + scores. Every foreground pixel
(`semantic argmax != bg`) votes `(x+dx, y+dy)` and is assigned to the **nearest** center -> instance
pixel groups (this is what splits touching same-class lesions). Instance class = majority semantic label
of its pixels; instance **confidence = center peak value** (the learned score). Polygon from the largest
external contour, normalized coords.

In [ ]:
def find_peaks(center_prob, thresh, kernel):
    t = torch.from_numpy(center_prob)[None, None]
    pooled = F.max_pool2d(t, kernel, stride=1, padding=kernel // 2)
    keep = (t == pooled) & (t > thresh)
    ys, xs = torch.where(keep[0, 0])
    ys = ys.numpy(); xs = xs.numpy()
    return [(int(y), int(x), float(center_prob[y, x])) for y, x in zip(ys, xs)]

@torch.no_grad()
def predict_small_instances(ip):
    img = load_image_resized(ip)
    x = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
    x = ((x - IMAGENET_MEAN) / IMAGENET_STD).unsqueeze(0).to(DEVICE)
    model.eval()
    sem_l, ctr_l, off_p = split_outputs(model(x))
    sem_prob = torch.softmax(sem_l, 1)[0].cpu().numpy()            # (N_SEM, H, W)
    ctr = torch.sigmoid(ctr_l)[0].cpu().numpy()                    # (H, W)
    off = (off_p[0].cpu().numpy()) * OFFSET_NORM                   # (2, H, W) in pixels
    sem_argmax = sem_prob.argmax(0)
    fg = sem_argmax != 0
    peaks = find_peaks(ctr, PEAK_THRESH, PEAK_NMS_KERNEL)
    if not peaks or int(fg.sum()) == 0: return []
    cy = np.array([p[0] for p in peaks], dtype=np.float64)
    cx = np.array([p[1] for p in peaks], dtype=np.float64)
    cs = np.array([p[2] for p in peaks], dtype=np.float64)
    fy, fx = np.where(fg)
    vx = fx + off[0, fy, fx]; vy = fy + off[1, fy, fx]            # voted centroid per fg pixel
    d2 = (vx[:, None] - cx[None, :]) ** 2 + (vy[:, None] - cy[None, :]) ** 2
    assign = d2.argmin(1)                                          # nearest center
    dets = []
    for k in range(len(peaks)):
        sel = assign == k
        if int(sel.sum()) < MIN_COMPONENT_PX: continue
        pys = fy[sel]; pxs = fx[sel]
        cl = int(np.bincount(sem_argmax[pys, pxs]).argmax())       # majority semantic class
        if cl == 0: continue
        m = np.zeros(fg.shape, dtype=np.uint8); m[pys, pxs] = 1
        cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not cnts: continue
        cnt = max(cnts, key=cv2.contourArea)
        if len(cnt) < 3: continue
        poly = cnt.reshape(-1, 2).astype(np.float64)
        poly[:, 0] /= IMG_W; poly[:, 1] /= IMG_H
        dets.append({"cls": int(sem_to_small[cl]), "conf": float(cs[k]), "poly": poly})
    return dets

inst_dets = {ip: predict_small_instances(ip) for ip in tqdm(val_objs, desc="instance-seg val")}
print("instance-seg instances:", sum(len(v) for v in inst_dets.values()), "over", len(inst_dets), "images")

## 7. YOLO (V6) for the LARGE classes

In [ ]:
MANUAL_V6_PATH = None
def find_pt(include, exclude=()):
    cands = []
    for p in Path("/kaggle/input").rglob("*.pt"):
        t = str(p).lower()
        if any(x in t for x in exclude): continue
        score = sum(10 for k in include if k in t) + (20 if p.name.lower().endswith("best.pt") else 0)
        if score > 0: cands.append((score, p))
    cands.sort(key=lambda z: z[0], reverse=True)
    return [p for _, p in cands]

v6 = find_pt(["version6", "v6"], exclude=["stage2", "version9", "version10", "version13", "version14", "version15"])
V6_PATH = Path(MANUAL_V6_PATH) if MANUAL_V6_PATH else (v6[0] if v6 else None)
print("V6:", V6_PATH)
if V6_PATH is None or not V6_PATH.exists():
    raise FileNotFoundError("No V6 detector found. Add version6_...best.pt or set MANUAL_V6_PATH.")
yolo = YOLO(str(V6_PATH))

def yolo_large_dets(ip):
    res = yolo.predict(source=ip, imgsz=768, conf=CAPTURE_CONF, iou=0.7, max_det=300,
                       device=(0 if DEVICE == "cuda" else "cpu"), task="segment", verbose=False, save=False)[0]
    out = []
    if res.masks is not None and res.boxes is not None and len(res.boxes) > 0:
        cls = res.boxes.cls.cpu().numpy().astype(int); conf = res.boxes.conf.cpu().numpy(); xyn = res.masks.xyn
        for i in range(min(len(cls), len(xyn))):
            if int(cls[i]) not in LARGE_IDX: continue                # large classes only
            poly = np.asarray(xyn[i], dtype=np.float64)
            if poly.ndim != 2 or len(poly) < 3: continue
            out.append({"cls": int(cls[i]), "conf": float(conf[i]), "poly": poly})
    return out

yolo_dets = {ip: yolo_large_dets(ip) for ip in tqdm(val_objs, desc="yolo large val")}
print("yolo large instances:", sum(len(v) for v in yolo_dets.values()))

## 8. Comparable Mask mAP — instance-seg-small vs V6 (and vs src/11), and the hybrid

Same matcher as src/03/04/09 (local-frame mask-IoU, greedy conf-ranked, 10 IoU thr, 101-pt AP).

In [ ]:
def poly_to_local_mask(poly_norm, w, h):
    pts = poly_norm.copy(); pts[:, 0] *= w; pts[:, 1] *= h
    gx0 = max(0, int(np.floor(pts[:,0].min()))); gy0 = max(0, int(np.floor(pts[:,1].min())))
    gx1 = min(w, int(np.ceil(pts[:,0].max())));  gy1 = min(h, int(np.ceil(pts[:,1].max())))
    gw = max(1, gx1 - gx0); gh = max(1, gy1 - gy0)
    m = np.zeros((gh, gw), dtype=np.uint8); pts[:, 0] -= gx0; pts[:, 1] -= gy0
    cv2.fillPoly(m, [pts.astype(np.int32)], 1)
    return (gx0, gy0, gx1, gy1), m

def iou_local(pbox, pmask, gbox, gmask):
    pa = int(pmask.sum()); ga = int(gmask.sum())
    if pa == 0 or ga == 0: return 0.0
    ix0 = max(pbox[0], gbox[0]); iy0 = max(pbox[1], gbox[1])
    ix1 = min(pbox[2], gbox[2]); iy1 = min(pbox[3], gbox[3]); inter = 0
    if ix1 > ix0 and iy1 > iy0:
        pc = pmask[iy0-pbox[1]:iy1-pbox[1], ix0-pbox[0]:ix1-pbox[0]]
        gc = gmask[iy0-gbox[1]:iy1-gbox[1], ix0-gbox[0]:ix1-gbox[0]]
        inter = int(np.logical_and(pc, gc).sum())
    union = pa + ga - inter
    return inter / union if union > 0 else 0.0

def ap_101(confs, tps, npos):
    if npos == 0: return float("nan")
    if not confs: return 0.0
    o = np.argsort(-np.asarray(confs)); tps = np.asarray(tps)[o]
    tp_c = np.cumsum(tps); fp_c = np.cumsum(1 - tps)
    rec = tp_c / npos; prec = tp_c / np.maximum(tp_c + fp_c, 1e-9)
    return sum((prec[rec >= r].max() if np.any(rec >= r) else 0.0) for r in np.linspace(0,1,101)) / 101.0

n_gt = np.zeros(num_classes, dtype=int)
for o in val_objs.values():
    for c, _ in o: n_gt[c] += 1
wh = {}
for ip in val_objs:
    im = cv2.imread(ip); wh[ip] = (im.shape[1], im.shape[0])
gt_local = {ip: [(c, *poly_to_local_mask(p, *wh[ip])) for c, p in o] for ip, o in val_objs.items()}

def score(dets_by_image):
    records = {(c, ti): [] for c in range(num_classes) for ti in range(len(IOU_THRESHOLDS))}
    for ip in val_objs:
        gts = gt_local[ip]
        pls = [(d["cls"], d["conf"], *poly_to_local_mask(d["poly"], *wh[ip])) for d in dets_by_image.get(ip, [])]
        for ti, thr in enumerate(IOU_THRESHOLDS):
            order = sorted(range(len(pls)), key=lambda i: pls[i][1], reverse=True)
            used = [False] * len(gts)
            for pi in order:
                pc, sc, pbox, pm = pls[pi]; best, bg = 0.0, -1
                for gi, (gc, gbox, gm) in enumerate(gts):
                    if used[gi] or gc != pc: continue
                    v = iou_local(pbox, pm, gbox, gm)
                    if v > best: best, bg = v, gi
                tp = 1 if (bg >= 0 and best >= thr) else 0
                if tp: used[bg] = True
                records[(pc, ti)].append((sc, tp))
    pac = np.full(num_classes, np.nan)
    for c in range(num_classes):
        if n_gt[c] == 0: continue
        pac[c] = np.nanmean([ap_101([r[0] for r in records[(c,ti)]],
                                    [r[1] for r in records[(c,ti)]], n_gt[c])
                             for ti in range(len(IOU_THRESHOLDS))])
    return pac

hybrid = {ip: yolo_dets[ip] + inst_dets[ip] for ip in val_objs}    # large->YOLO + small->instance seg
ap_inst = score(inst_dets)                                         # instance seg, small classes only
ap_hyb  = score(hybrid)

def agg(pac, idxs):
    vals = [pac[c] for c in idxs if not np.isnan(pac[c])]
    return float(np.mean(vals)) if vals else float("nan")

print("per-class Mask AP — instance-seg (small) vs src/11 semseg vs V6:")
print(f"{'class':>14s}{'n_gt':>6s}{'inst':>9s}{'semseg11':>10s}{'V6_ref':>9s}{'d(V6)':>9s}")
for c in SMALL_IDX:
    key = _norm_class_key(names[c]); ref = V6_REF_AP.get(key, float('nan'))
    s11 = SEMSEG_REF_SMALL.get(key, float('nan'))
    tag = "" if key in SUPPORTED_SMALL else "  (noise)"
    print(f"{names[c]:>14s}{int(n_gt[c]):>6d}{ap_inst[c]:>9.3f}{s11:>10.3f}{ref:>9.3f}{ap_inst[c]-ref:>+9.3f}{tag}")

sup_idx = [c for c in SMALL_IDX if _norm_class_key(names[c]) in SUPPORTED_SMALL]
inst_sup = agg(ap_inst, sup_idx)
v6_sup   = float(np.mean([V6_REF_AP[_norm_class_key(names[c])] for c in sup_idx]))
s11_sup  = float(np.mean([SEMSEG_REF_SMALL[_norm_class_key(names[c])] for c in sup_idx]))
print()
print(f"HEADLINE  supported-small (caries1/2/3/5): inst {inst_sup:.4f}  vs semseg(src11) {s11_sup:.4f}  vs V6 {v6_sup:.4f}")
print(f"          inst - semseg = {inst_sup-s11_sup:+.4f}   |   inst - V6 = {inst_sup-v6_sup:+.4f}")
print(f"HYBRID    aggregate (9 classes):           {agg(ap_hyb, range(num_classes)):.4f}  vs V6 {V6_REF_MAP:.4f}  ({agg(ap_hyb, range(num_classes))-V6_REF_MAP:+.4f})")

## 9. Save + how to read

- **`instance_seg_hybrid_baseline.csv`** = per-class instance-seg-small AP + hybrid AP + (src/11 semseg, V6) refs.
- **`instance_seg_small_baseline.pt`** = the trained center+offset segmenter, for follow-ups.
- Save the CSV as `results/versionN_results.csv` to keep the durable per-run record.

**Decision (pre-registered):**
- **`inst` headline clearly beats src/11 semseg AND V6 supported-small** (beyond ~0.003) -> route B's
  learned grouping + score is the right fix; refine (binary+subtype head, higher resolution / tiling,
  a center-AP checkpoint metric, semseg/center TTA) and build a test-set submission path.
- **`inst` beats src/11 but not V6** -> the grouping+score machinery helped (the diagnosis was right)
  but the pixel signal (resolution) still caps it -> next single-variable lever is resolution / tiling.
- **`inst` flat vs src/11** -> the instance/score machinery was not the bottleneck; the small-class
  deficit is the pixel signal -> resolution/tiling, or stop and keep the 2-model ensemble (LB 0.31753).
- The **hybrid aggregate** stays the conservative number (small classes are low-weight); judge the
  *line* on the headline, judge a *submission* on the aggregate / the leaderboard.

In [ ]:
OUT = Path("/kaggle/working")
rows = []
for c in range(num_classes):
    rows.append({"class": names[c], "n_gt": int(n_gt[c]),
                 "inst_small_AP": (None if np.isnan(ap_inst[c]) else float(ap_inst[c])),
                 "hybrid_AP": (None if np.isnan(ap_hyb[c]) else float(ap_hyb[c])),
                 "semseg11_ref": SEMSEG_REF_SMALL.get(_norm_class_key(names[c])),
                 "V6_ref": V6_REF_AP.get(_norm_class_key(names[c]))})
pd.DataFrame(rows).to_csv(OUT / "instance_seg_hybrid_baseline.csv", index=False)
torch.save(model.state_dict(), OUT / "instance_seg_small_baseline.pt")
print("saved /kaggle/working/instance_seg_hybrid_baseline.csv + instance_seg_small_baseline.pt")